# Study-basin pipeline — DPWM baseline + EURO-CORDEX future scenario

This notebook runs the **full thesis workflow** for the six study catchments, from historical meteorology to future water deficits. Each step calls the same Python modules as the standalone scripts; the DPWM equations and parameters are unchanged.

**What this pipeline does (big picture)**

1. **Historical period** — Build model forcing from eStreams, calibrate DPWM against observed discharge, simulate baseline streamflow, derive flow thresholds.
2. **Future period** — Bias-correct climate model data, build future forcing, run DPWM with **the same calibrated parameters**, compare future flow to baseline thresholds (deficits).

**How to use this notebook**

| Action | What to do |
|--------|------------|
| **Pilot (recommended first)** | In the config cell: `BASINS = ["SE000197"]` (Stensjön / Gothenburg) |
| **All six basins** | Set `BASINS = ALL_STUDY_BASINS` when the pilot works |
| **Run order** | Config → Setup → Step 1 → … → Step 7 (or *Run All*) |
| **First calibration** | Set `RUN_STEP2_CALIBRATION = True` once (slow). Later set `False` and reuse the CSV |
| **Fast test** | Keep all `RUN_*_PLOTS` / `SAVE_RAINPLUSMELT_DIAGNOSTICS` flags `False` |

**Study basins (full list):** CH000127, DESN1585, FR002785, GB000215, SE000197, ES000700

**Batch (~3750 basins):** use [`../batch_running/batch_baseline_thresholds.ipynb`](../batch_running/batch_baseline_thresholds.ipynb).

**External data you must have on disk:** eStreams CSVs under `data/csv_estream/`, streamflow under `data/streamflow/`, parameter bounds in `data/parameters/Parameters_basins.dbf`, and EURO-CORDEX NetCDFs under `CORDEX_BASE` (default `F:\Data\EUROCORDEX_MPI`).


In [96]:
# --- CONFIG ---
from pathlib import Path

ALL_STUDY_BASINS = [
    "CH000127", "DESN1585", "FR002785", "GB000215", "SE000197", "ES000700"
]
BASINS = ["BEWA0040"]

EXTRACTION = "areal"
CORDEX_BASE = Path(r"F:\Data\EUROCORDEX_MPI")

RUN_STEP1_HISTORICAL_FORCING = True
RUN_STEP2_CALIBRATION = False
RUN_STEP3_BASELINE_DPWM = True
RUN_STEP4_THRESHOLDS = True
RUN_STEP5_BIAS_CORRECTION = True
RUN_STEP6_FUTURE_DPWM = True
RUN_STEP7_DEFICITS = True

SAVE_RAINPLUSMELT_DIAGNOSTICS = False
RUN_BASELINE_HYDROGRAPHS = False
RUN_DEFICIT_PLOTS = True
RUN_DEFICIT_FREQUENCY_PLOTS = True

BC_WORKERS = 1
SKIP_EXISTING_BIAS = True


**Configuration cell - what each flag does**

| Variable | Meaning |
|----------|---------|
| `BASINS` | Catchments to process in **this run** (start with `["SE000197"]`) |
| `ALL_STUDY_BASINS` | Full six-basin list for production runs |
| `EXTRACTION` | `"areal"` (catchment-weighted CORDEX) or `"nearest"` (single grid cell) |
| `CORDEX_BASE` | Root folder with `historical/` and `future/` NetCDF subfolders |
| `RUN_STEP1` … `RUN_STEP7` | Turn individual pipeline steps on/off |
| `RUN_STEP2_CALIBRATION` | **Off** after first successful calibration (saves hours) |
| `SAVE_RAINPLUSMELT_DIAGNOSTICS` | Extra snow/rain diagnostic CSV + PNG per basin (Step 1) |
| `RUN_BASELINE_HYDROGRAPHS` | PNG hydrographs after Step 3 |
| `RUN_DEFICIT_PLOTS` | PNG annual/seasonal deficit-summary plots after Step 7 |
| `RUN_DEFICIT_FREQUENCY_PLOTS` | PNG deficit-frequency curves (D mm vs probability) after Step 7 |
| `BC_WORKERS` | Parallel processes for Step 5 (use `1` for one basin) |
| `SKIP_EXISTING_BIAS` | Skip Step 5 for basins that already have BC outputs |

**Prerequisite chain:** Step 1 → 2 → 3 → 4 (baseline branch) and Step 1 → 5 → 6 → 7 (future branch). Step 2 can be skipped only if calibration CSV already exists for your `BASINS`.

In [97]:
import os
import sys
from pathlib import Path

# notebooks/project_root.py (works from notebooks/single_basin_running/ too)
_nb = Path.cwd().resolve()
for _p in [_nb, *_nb.parents]:
    if (_p / "project_root.py").is_file() and _p.name == "notebooks":
        if str(_p) not in sys.path:
            sys.path.insert(0, str(_p))
        break
from project_root import find_project_root

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import calibration_io  # after sys.path — lives in project root

calibration_io.BASIN_IDS = list(BASINS)
BASINS_CSV = ",".join(BASINS)

suffix = "" if EXTRACTION == "areal" else "_nearest"
PATHS = {
    "bc": ROOT / f"Results/eurocordex_bias_corrected{suffix}",
    "forcing_future": ROOT / f"Results/forcing_future{suffix}",
    "discharge_future": ROOT / f"Results/discharge_components_future{suffix}",
    "deficit": ROOT / f"Results/deficit_q80{suffix}",
}
print("ROOT", ROOT, "\nBASINS", BASINS)


ROOT C:\Users\eiv001\Desktop\thesis\Cursor 
BASINS ['BEWA0040']


## Step 1 - Historical forcing (`rainplusmelt` + PET)

**Purpose**
Convert daily eStreams meteorology into the **two DPWM input time series** used in every simulation:
- **rainplusmelt** — liquid precipitation + snowmelt (mm/day)
- **PET** — potential evapotranspiration (mm/day), taken from eStreams `pet_mean`

**Process (what the code does)**
1. Read `frac_snow` per basin from `estreams_hydrometeo_signatures.csv`.
2. For each day and basin:
   - If **T > 0 °C**: all precipitation is rain.
   - If **T ≤ 0 °C**: split precipitation into snow and rain using `frac_snow`.
   - Track a **snowpack**; melt when T > 0 and snow exists: `melt = min(T × P × DDF, snowpack)` with **DDF = 5.9** mm °C⁻¹ day⁻¹.
   - **rainplusmelt = rain + melt**.
3. Stack all basins into wide tables (`date` + one column per basin) and save CSV + MATLAB `.mat` files.

**Inputs**
| Item | Path / note |
|------|-------------|
| Daily meteo per basin | `data/csv_estream/estreams_meteorology_{basin_id}.csv` (columns include `date`, `p_mean`, `t_mean`, `pet_mean`) |
| Snow fraction | `data/csv_estream/estreams_hydrometeo_signatures.csv` (`basin_id`, `frac_snow`) |
| Basin list | `BASINS` in config cell |

**Outputs**
| File | Description |
|------|-------------|
| `Results/forcing/rainplusmelt.csv` | Wide CSV: `date` + `{basin_id}` columns |
| `Results/forcing/PET.csv` | Same layout for PET |
| `Matlab_files/rainplusmelt.mat`, `PET.mat` | MATLAB-compatible matrices |
| `Results/diagnostics/diagnostics_{basin}.csv` | Optional per-basin breakdown (if `SAVE_RAINPLUSMELT_DIAGNOSTICS = True`) |
| `Results/diagnostics/diagnostics_{basin}.png` | Optional diagnostic figure |

**Config flags**
- `RUN_STEP1_HISTORICAL_FORCING` — run or skip this step
- `SAVE_RAINPLUSMELT_DIAGNOSTICS` — extra CSV/PNG per basin (slower)

**Used by**
Steps 2, 3 (baseline DPWM).

> **Before running the code cell below:** run **CONFIG** then **SETUP** (two cells above).  
> If you only run Step 1, nothing happens or you get `Skipped step 1` / `NameError`.


In [98]:
# Step 1 — run CONFIG + SETUP cells first!
from datetime import datetime
from pathlib import Path

_required = ("BASINS", "RUN_STEP1_HISTORICAL_FORCING")
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing {_missing}. Run the CONFIG cell (and SETUP cell) above, then run this cell again."
    )

if not RUN_STEP1_HISTORICAL_FORCING:
    print("Step 1 NOT run: RUN_STEP1_HISTORICAL_FORCING is False in CONFIG.")
    print("Set RUN_STEP1_HISTORICAL_FORCING = True and re-run CONFIG, then this cell.")
else:
    # Ensure project root even if SETUP was skipped
    if "ROOT" not in globals():
        import os
        import sys
        _nb = Path.cwd().resolve()
        for _p in [_nb, *_nb.parents]:
            if (_p / "project_root.py").is_file() and _p.name == "notebooks":
                if str(_p) not in sys.path:
                    sys.path.insert(0, str(_p))
                break
        from project_root import find_project_root
        ROOT = find_project_root()
        os.chdir(ROOT)
        if str(ROOT) not in sys.path:
            sys.path.insert(0, str(ROOT))
        import calibration_io
        calibration_io.BASIN_IDS = list(BASINS)

    from prepare_eastream_inputs import main as step1_main

    print("=" * 60)
    print("STEP 1 — historical forcing")
    print("Basins:", BASINS)
    print("Project root:", Path.cwd().resolve())
    print("=" * 60)

    step1_main(
        basin_ids=list(BASINS),
        save_diagnostic_plots=bool(globals().get("SAVE_RAINPLUSMELT_DIAGNOSTICS", False)),
    )

    print("\n--- Output check ---")
    for name in ("rainplusmelt.csv", "PET.csv"):
        p = Path("Results/forcing") / name
        if p.exists():
            ts = datetime.fromtimestamp(p.stat().st_mtime)
            print(f"OK   {p.resolve()}")
            print(f"     size={p.stat().st_size:,} bytes  last modified={ts:%Y-%m-%d %H:%M:%S}")
        else:
            print(f"MISSING   {p.resolve()}")


STEP 1 — historical forcing
Basins: ['BEWA0040']
Project root: C:\Users\eiv001\Desktop\thesis\Cursor
Project root: C:\Users\eiv001\Desktop\thesis\Cursor
Processing basin BEWA0040...
Shapes:
  rainplusmelt: (27210, 1)
  PET: (27210, 1)
Wrote C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\rainplusmelt.csv
Wrote C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\PET.csv
Wrote C:\Users\eiv001\Desktop\thesis\Cursor\Matlab_files\rainplusmelt.mat
Wrote C:\Users\eiv001\Desktop\thesis\Cursor\Matlab_files\PET.mat
Diagnostics -> C:\Users\eiv001\Desktop\thesis\Cursor\Results\diagnostics/

--- Output check ---
OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\rainplusmelt.csv
     size=453,496 bytes  last modified=2026-06-02 14:32:05
OK   C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\PET.csv
     size=459,927 bytes  last modified=2026-06-02 14:32:05


## Step 2 - Calibrate DPWM parameters (historical)

### Purpose
Find the **best DPWM parameter set** for each basin by comparing simulated discharge to **observed** streamflow over the historical calibration window.

### Process (what the code does)
1. Load `rainplusmelt` and `PET` from Step 1.
2. Load observed discharge from the eStreams streamflow archive; convert **m³/s → mm/day** using catchment area from `Parameters_basins.dbf`.
3. For each basin, run a **particle-swarm optimisation** (PSO) that:
   - Simulates DPWM for ~4018 days (with 365-day warm-up).
   - Maximises **KGE on inverse flow** `KGE(1/(Q+1.5))` (objective `invq_only`, formulation KGE 2012).
   - Searches within parameter bounds from the DBF (e.g. `Sm` in [0, 8000] mm).
4. Split valid days 70% calibration / 30% validation; write metrics and optimal parameters to CSV.

### Inputs (required)
| Item | Path / note |
|------|-------------|
| Forcing | `Results/forcing/rainplusmelt.csv`, `PET.csv` (Step 1) |
| Observed Q | `data/streamflow/estreams_timeseries_streamflow_v02.csv` |
| Parameter bounds | `data/parameters/Parameters_basins.dbf` |
| Signatures | `data/csv_estream/estreams_hydrometeo_signatures.csv` |

### Outputs (generated)
| File | Description |
|------|-------------|
| `Results/calibration/calibration_validation_results.csv` | Optimal parameters + KGE metrics per basin |
| `Results/calibration/calibration_validation_results_invq_only.csv` | May be created depending on run mode |
| `Results/cache_obs/` | Cached aligned observations (`.npz`) |

### Runtime
**Very slow** (tens of minutes to hours per basin depending on `max_gens`).  
→ First run: `RUN_STEP2_CALIBRATION = True`.  
→ Later runs: `False` if the CSV already contains your basins.

### Used by
Steps 3 and 6 (same calibrated parameters for baseline and future).


In [99]:
if RUN_STEP2_CALIBRATION:
    import calibrate_validate_dpwm as cal
    sys.argv = ['notebook', '--basins', BASINS_CSV, '--objective_target', 'invq_only',
                '--kge_mode', 'kge2012', '--inv_flow_offset', '1.5', '--q_obs_unit', 'm3s']
    cal.main()
else:
    print('Skipped step 2')


Skipped step 2


## Step 3 — Baseline simulation (historical DPWM)

### Purpose
Run the **calibrated** DPWM on historical forcing to obtain **baseline streamflow** and internal fluxes for the calibration period.

### Process (what the code does)
1. Read calibrated parameters for each basin from the Step 2 CSV (prefers `invq_only` results if available).
2. For each basin, run `DPWM.simulate(rainplusmelt, PET)` with a **365-day warm-up**.
3. Save total discharge and components (quick/slow flow, ET, etc.) to CSV.
4. Optionally plot hydrographs (`RUN_BASELINE_HYDROGRAPHS = True`).

### Inputs (required)
| Item | Path / note |
|------|-------------|
| Forcing | `Results/forcing/rainplusmelt.csv`, `PET.csv` |
| Calibration | `Results/calibration/calibration_validation_results*.csv` (Step 2) |
| Basin list | `BASINS` (must match columns in forcing + rows in calibration CSV) |

### Outputs (generated)
| File | Description |
|------|-------------|
| `Results/discharge_components/Q_total_all_basins.csv` | Wide CSV: daily total Q (mm/day) per basin |
| `Results/discharge_components/discharge_components_{basin}.csv` | Per-basin flux breakdown |
| `Results/hydrographs/` | Optional PNGs if `RUN_BASELINE_HYDROGRAPHS = True` |

### Used by
Step 4 (thresholds are computed from this baseline Q).


In [100]:
if RUN_STEP3_BASELINE_DPWM:
    import importlib
    import pandas as pd
    import calibration_io

    # Always sync basin list to current CONFIG before running step 3.
    calibration_io.BASIN_IDS = list(BASINS)

    # Decide which route to use:
    # - study route (run_dpwm): basins available in six-basin forcing + calibration CSV
    # - batch route: basins available in forcing_per_basin + calibration_batch params JSON
    rain_main = ROOT / 'Results' / 'forcing' / 'rainplusmelt.csv'
    main_cols = pd.read_csv(rain_main, nrows=1).columns.tolist() if rain_main.exists() else ['date']
    in_main_forcing = all(b in main_cols for b in BASINS)

    cal_df, _ = calibration_io.load_preferred_calibration_df(ROOT)
    cal_set = set(cal_df['basin_id'].astype(str))
    in_main_cal = all(b in cal_set for b in BASINS)

    if in_main_forcing and in_main_cal:
        import run_dpwm
        importlib.reload(run_dpwm)
        sys.argv = ['notebook'] + (['--make_plots'] if RUN_BASELINE_HYDROGRAPHS else [])
        run_dpwm.main()
    else:
        # Batch fallback for non-study basins (e.g. BEWA0040)
        from batch_baseline_workflow import simulate_baseline_q

        q_out_dir = ROOT / 'Results' / 'discharge_components'
        q_out_dir.mkdir(parents=True, exist_ok=True)

        q_wide = None
        for b in BASINS:
            df_b = simulate_baseline_q(
                b,
                forcing_dir=ROOT / 'Results' / 'forcing_per_basin',
                params_dir=ROOT / 'Results' / 'calibration_batch' / 'params',
                warm_span_days=365,
            )
            q_col = df_b.rename(columns={'Q_total': b})
            q_wide = q_col if q_wide is None else q_wide.merge(q_col, on='date', how='inner')

        q_path = q_out_dir / 'Q_total_all_basins.csv'
        q_wide.to_csv(q_path, index=False)
        print('Step 3 used batch fallback (forcing_per_basin + calibration_batch params).')
        print(f'Saved combined total discharge: {q_path}')
else:
    print('Skipped step 3')


Step 3 used batch fallback (forcing_per_basin + calibration_batch params).
Saved combined total discharge: C:\Users\eiv001\Desktop\thesis\Cursor\Results\discharge_components\Q_total_all_basins.csv


## Step 4 — Seasonal flow thresholds (Q20, Q50, Q80, Q90)

### Purpose
Define **variable daily flow thresholds** from the **baseline simulated discharge** (not observations). These thresholds represent “normal” flow for each day-of-year and are used later to quantify **deficits** in the future period.

### Process (what the code does)
1. Load `Q_total_all_basins.csv` from Step 3.
2. For each basin, keep only the **most recent 30 years** of simulated Q.
3. For each calendar day (1–366), compute quantiles **Q20, Q50, Q80, Q90** using a **30-day moving window** (±15 days across all years in that window).
4. Optionally smooth thresholds along the year.
5. Write one threshold file per basin (Q80 is used in Step 7).

### Inputs (required)
| Item | Path / note |
|------|-------------|
| Baseline discharge | `Results/discharge_components/Q_total_all_basins.csv` |

### Outputs (generated)
| File | Description |
|------|-------------|
| `Results/thresholds/` | Per-basin CSVs with DOY-based Q20, Q50, Q80, Q90 |
| e.g. `Results/thresholds/Q80_variable_last30y_{basin}.csv` | Daily Q80 threshold series |

### Important
Thresholds describe **historical model climatology**. Future deficits compare **future simulated Q** (Step 6) against this reference — the climate-change signal is in the future simulation, not in redefining Q80 from future Q.

### Used by
Step 7 (deficit calculation).


In [101]:
if RUN_STEP4_THRESHOLDS:
    import importlib
    import compute_seasonal_thresholds as thr
    importlib.reload(thr)
    sys.argv = ['notebook', '--q_csv', str(ROOT / 'Results' / 'discharge_components' / 'Q_total_all_basins.csv')]
    thr.main()
else:
    print('Skipped step 4')


Input: C:\Users\eiv001\Desktop\thesis\Cursor\Results\discharge_components\Q_total_all_basins.csv
Date window used: 1976-01-01 to 2005-12-31
Smoothing: circular 30-day moving average
Saved: Results\thresholds\seasonal_thresholds_q20_q50_q80_q90_last30y.csv
Saved: Results\thresholds\Q20_daily_thresholds_last30y.csv
Saved: Results\thresholds\Q50_daily_thresholds_last30y.csv
Saved: Results\thresholds\Q80_daily_thresholds_last30y.csv
Saved: Results\thresholds\Q90_daily_thresholds_last30y.csv
Saved: Results\thresholds\mean_q_daily_all_basins_last30y.csv


## Step 5 — Bias correction of EURO-CORDEX (future forcing)

### Purpose
Adjust **raw climate-model** daily series so they match **observed eStreams statistics** in a reference period, while **preserving the scenario trend**. The corrected series are the meteorological basis for future `rainplusmelt` and PET.

### Process (what the code does)
1. **Extract** CORDEX fields (pr, tas, tasmin, tasmax) for each catchment:
   - `EXTRACTION = "areal"` — area-weighted mean over grid cells overlapping the catchment polygon (default, thesis method).
   - `EXTRACTION = "nearest"` — single nearest grid cell (faster, for comparison runs).
2. **Reference period** (default): 1976–2005 — compare CORDEX *historical* experiment to eStreams observations.
3. Compute **correction factors per day-of-year** (30-day centred window):
   - Precipitation: **multiplicative** (wet-day sums).
   - Temperature: **additive**.
4. Apply factors to the CORDEX **scenario** period → bias-corrected daily series.
5. Compute **future PET** (Hargreaves) from corrected temperatures.
6. Enforce **tasmin ≤ tasmax** and recompute PET if needed.

### Inputs (required)
| Item | Path / note |
|------|-------------|
| CORDEX NetCDF tree | `CORDEX_BASE` → `historical/{pr,tas,...}/*.nc`, `future/{pr,tas,...}/*.nc` |
| eStreams observations | `data/csv_estream/estreams_meteorology_{basin}.csv` |
| Basin locations | `data/csv_estream/basins.csv` |
| Catchment polygons (areal only) | eStreams shapefile (path set inside `bias_correction.py`) |

### Outputs (generated, under `PATHS["bc"]`)
| File | Description |
|------|-------------|
| `pr_bc_daily_{basin}.csv` | Columns: `date`, `pr_raw`, `pr_bc` |
| `tas_bc_daily_{basin}.csv` | Raw + bias-corrected mean temperature |
| `tasmin_bc_daily_{basin}.csv`, `tasmax_bc_daily_{basin}.csv` | Min/max temperature |
| `pet_future_bc_{basin}.csv` | Columns: `date`, `pet_mm_day` |
| `factors_{var}_{basin}.csv` | DOY correction factors (diagnostic) |
| `grid_weights_{basin}.csv` | Areal extraction weights (cached) |

### Config flags
- `BC_WORKERS` — parallel basins (use `1` for single-basin pilot)
- `SKIP_EXISTING_BIAS` — skip basin if pr/tas/pet outputs already exist

### Runtime
**Long** (reads many NetCDF files). Ensure `F:\Data\EUROCORDEX_MPI` (or your `CORDEX_BASE`) is available.

### Used by
Step 6.


In [ ]:
if RUN_STEP5_BIAS_CORRECTION:
    import bias_correction as bc
    cmd = ['notebook', '--basins', BASINS_CSV, '--extraction', EXTRACTION,
           '--cordex_base', str(CORDEX_BASE), '--out_dir', str(PATHS['bc']),
           '--workers', str(BC_WORKERS)]
    if SKIP_EXISTING_BIAS:
        cmd.append('--skip_existing')
    sys.argv = cmd
    bc.main()
else:
    print('Skipped step 5')


Basins (1): BEWA0040
Reference period: 1976-01-01 to 2005-12-31 (window = 30 days)
Spatial extraction: areal
[BEWA0040] lat=50.2989, lon=4.1751, extraction=areal
  Areal extraction: 4 cells overlap catchment (weights -> grid_weights_BEWA0040.csv)
[BEWA0040] pr: 6+19 NC files
[BEWA0040] tas: 6+19 NC files
[BEWA0040] tasmin: 6+19 NC files
[BEWA0040] tasmax: 6+19 NC files


## Step 6 — Future forcing + DPWM simulation

### Purpose
1. Build **future** `rainplusmelt` and `PET` from bias-corrected climate data (same snow routine as Step 1).
2. Run DPWM with **the same calibrated parameters as the baseline** (Step 2) — only forcing changes, not parameters.

### Process (what the code does)
1. Read bias-corrected `pr_bc`, temperatures, and `pet_future_bc` per basin from Step 5.
2. Apply the **snowpack / melt model** (`frac_snow`, DDF = 5.9) → daily `rainplusmelt`; use corrected PET series.
3. Write wide forcing CSVs under `PATHS["forcing_future"]`.
4. Simulate DPWM for the full future period (with warm-up) → future discharge time series.

### Inputs (required)
| Item | Path / note |
|------|-------------|
| Bias-corrected meteo | `PATHS["bc"]/pr_bc_daily_*.csv`, `tas_bc_daily_*.csv`, `pet_future_bc_*.csv` |
| Snow fraction | `data/csv_estream/estreams_hydrometeo_signatures.csv` |
| Calibrated parameters | Step 2 calibration CSV |

### Outputs (generated)
| File | Description |
|------|-------------|
| `PATHS["forcing_future"]/rainplusmelt.csv` | Future forcing (wide format) |
| `PATHS["forcing_future"]/PET.csv` | Future PET |
| `PATHS["discharge_future"]/Q_total_all_basins.csv` | Future total simulated Q |
| `PATHS["discharge_future"]/discharge_components_{basin}.csv` | Per-basin future fluxes |

### Note on folder suffix
If `EXTRACTION = "nearest"`, outputs go to `*_nearest` folders (e.g. `Results/forcing_future_nearest/`) so areal and nearest runs do not overwrite each other.

### Used by
Step 7.


In [ ]:
if RUN_STEP6_FUTURE_DPWM:
    import run_future_dpwm as fut
    sys.argv = ['notebook', '--root', str(ROOT), '--basins', BASINS_CSV,
                '--bc_dir', str(PATHS['bc']), '--forcing_out', str(PATHS['forcing_future']),
                '--discharge_out', str(PATHS['discharge_future'])]
    fut.main()
else:
    print('Skipped step 6')


## Step 7 — Water deficits (future Q compared to baseline Q80)

### Purpose
Quantify **low-flow deficits** in the **future simulation**: when and how much daily simulated discharge falls below the **baseline Q80 threshold** (from Step 4).

### Process (what the code does)
1. Load **future** discharge `Q_total_all_basins.csv` (Step 6).
2. Load **baseline Q80** threshold series per basin (Step 4), mapped by day-of-year onto future dates.
3. For each day with Q < Q80: deficit **D = Q80 − Q** (else D = 0).
4. Aggregate by **hydrological year** (starts 1 October) and by **meteorological season** (DJF, MAM, JJA, SON):
   - total ΣD per year/season,
   - number of deficit days,
   - mean deficit intensity.
5. Optionally plot per-basin figures (`RUN_DEFICIT_PLOTS = True`).

### Inputs (required)
| Item | Path / note |
|------|-------------|
| Future discharge | `PATHS["discharge_future"]/Q_total_all_basins.csv` |
| Q80 thresholds | `Results/thresholds/` (from Step 4) |

### Outputs (generated, under `PATHS["deficit"]`)
| File | Description |
|------|-------------|
| `deficits_hydro_year_{basin}.csv` | Annual ΣD and deficit-day statistics |
| `deficits_season_{basin}.csv` | Seasonal ΣD (DJF/MAM/JJA/SON) |
| `plots/deficit_hydro_{basin}.png` | Optional if `RUN_DEFICIT_PLOTS = True` |

### Interpretation
- **Larger ΣD** → more severe or frequent low-flow conditions vs baseline climatology.
- Thresholds are fixed from **historical model Q**; future changes come from **future forcing + same parameters**.

### Thesis link
Supports the “future scenario” analysis: climate input → model Q → comparison to baseline flow standards.


In [ ]:
if RUN_STEP7_DEFICITS:
    import importlib
    import compute_q80_deficits as defc
    importlib.reload(defc)
    cmd = ['notebook', '--root', str(ROOT), '--basins', BASINS_CSV,
           '--q_csv', str(PATHS['discharge_future'] / 'Q_total_all_basins.csv'),
           '--out_dir', str(PATHS['deficit'])]
    if RUN_DEFICIT_PLOTS:
        cmd.append('--plot')
    if RUN_DEFICIT_FREQUENCY_PLOTS:
        cmd.append('--plot_frequency')
    sys.argv = cmd
    defc.main()
else:
    print('Skipped step 7')


## Optional — Plots and diagnostics (run separately)

This pipeline is intentionally **CSV-first**. Figures are slow and are not required for the numerical results above.

| If you want to… | Use | Needs these steps completed |
|-----------------|-----|----------------------------|
| Check snow → rain+melt split | `prepare_eastream_inputs.ipynb` | Step 1 |
| Baseline hydrographs (obs vs sim) | `run_hydro_plots.ipynb` or `RUN_BASELINE_HYDROGRAPHS` | Steps 1–3 |
| Future hydrograph + Q20/Q50/Q80 lines | `plot_future_hydrograph.ipynb` | Steps 4, 6 |
| Compare areal vs nearest extraction | `compare_extraction_methods.ipynb` | Full pipeline ×2 (different `EXTRACTION`) |

**Recommendation:** finish Steps 1–7 for `SE000197` first, verify output files in the summary cell below, then enable plot notebooks only when you need figures for the thesis.


In [16]:
for p in [ROOT/'Results/forcing/rainplusmelt.csv', ROOT/'Results/forcing/PET.csv',
          PATHS['discharge_future']/'Q_total_all_basins.csv']:
    print('OK ' if p.exists() else 'MISSING', p)


OK  C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\rainplusmelt.csv
OK  C:\Users\eiv001\Desktop\thesis\Cursor\Results\forcing\PET.csv
OK  C:\Users\eiv001\Desktop\thesis\Cursor\Results\discharge_components_future\Q_total_all_basins.csv
